<a href="https://colab.research.google.com/github/West-Ben/ElaraLM/blob/main/GoEmotionFineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -q transformers datasets scikit-learn accelerate

In [2]:
import torch

assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score
import torch

In [7]:

# Original 28 label names
label_names_28 = raw["train"].features["labels"].feature.names

# Merge map: source -> target (by name)
MERGE_MAP = {
    "grief":  "sadness",
    "relief": "joy",
    "pride":  "admiration",
}

# Build new 25-label set (preserve original order, drop merged sources)
label_names_25 = [l for l in label_names_28 if l not in MERGE_MAP]

# Index lookups
old_to_idx = {name: i for i, name in enumerate(label_names_28)}
new_to_idx = {name: i for i, name in enumerate(label_names_25)}

# Map each old index to new index (merged classes point to target's new index)
old_idx_to_new_idx = {}
for old_name in label_names_28:
    target_name = MERGE_MAP.get(old_name, old_name)  # remap or keep
    old_idx_to_new_idx[old_to_idx[old_name]] = new_to_idx[target_name]

NUM_LABELS_25 = len(label_names_25)

print(f"Labels: 28 -> {NUM_LABELS_25}")
print("\nMerges applied:")
for src, tgt in MERGE_MAP.items():
    print(f"  {src} -> {tgt}")
print("\nNew label set:")
for i, name in enumerate(label_names_25):
    print(f"  {i:2d}  {name}")

Labels: 28 -> 25

Merges applied:
  grief -> sadness
  relief -> joy
  pride -> admiration

New label set:
   0  admiration
   1  amusement
   2  anger
   3  annoyance
   4  approval
   5  caring
   6  confusion
   7  curiosity
   8  desire
   9  disappointment
  10  disapproval
  11  disgust
  12  embarrassment
  13  excitement
  14  fear
  15  gratitude
  16  joy
  17  love
  18  nervousness
  19  optimism
  20  realization
  21  remorse
  22  sadness
  23  surprise
  24  neutral


In [13]:
def preprocess_merged(batch):
    encoding = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    multi_hot = np.zeros((len(batch["labels"]), NUM_LABELS_25), dtype=np.float32)
    for i, label_list in enumerate(batch["labels"]):
        for old_idx in label_list:
            new_idx = old_idx_to_new_idx[old_idx]
            multi_hot[i][new_idx] = 1.0  # OR logic — if grief+sadness both present, still 1.0
    encoding["labels"] = multi_hot.tolist()
    return encoding

tokenized_25 = raw.map(
    preprocess_merged,
    batched=True,
    remove_columns=raw["train"].column_names
)
tokenized_25.set_format("torch")

print("Label shape check:", tokenized_25["train"][0]["labels"].shape)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Label shape check: torch.Size([25])


In [14]:
model_25 = AutoModelForSequenceClassification.from_pretrained(
    "distilroberta-base",
    num_labels=NUM_LABELS_25,
    problem_type="multi_label_classification",
)
model_25.to("cuda")
print(f"Parameters: {sum(p.numel() for p in model_25.parameters()) / 1e6:.1f}M")

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 82.1M


In [15]:
RETRAIN_25_DIR = "go_emotions_model_25"

training_args_25 = TrainingArguments(
    output_dir=RETRAIN_25_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    logging_steps=100,
    save_total_limit=2,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
def compute_metrics_25(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    labels = labels.astype(int)
    return {
        "f1_macro":    f1_score(labels, preds, average="macro",    zero_division=0),
        "f1_micro":    f1_score(labels, preds, average="micro",    zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

In [20]:
from transformers import DataCollatorWithPadding

class MultiLabelDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features)
        if "labels" in batch:
            batch["labels"] = batch["labels"].to(torch.float32)
        return batch

data_collator = MultiLabelDataCollator(tokenizer=tokenizer)

trainer_25 = Trainer(
    model=model_25,
    args=training_args_25,
    train_dataset=tokenized_25["train"],
    eval_dataset=tokenized_25["validation"],
    compute_metrics=compute_metrics_25,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    data_collator=data_collator
)

trainer_25.train()

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,F1 Weighted
1,0.120293,0.111951,0.221847,0.477394,0.392625


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,F1 Weighted
1,0.120293,0.111951,0.221847,0.477394,0.392625
2,0.099232,0.098625,0.384789,0.534238,0.489404
3,0.090883,0.094227,0.437099,0.568299,0.533213
4,0.086841,0.093834,0.456039,0.572125,0.539922
5,0.081498,0.093977,0.479079,0.578619,0.552719


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6785, training_loss=0.11429129918884862, metrics={'train_runtime': 770.5519, 'train_samples_per_second': 281.681, 'train_steps_per_second': 8.805, 'total_flos': 7190960543424000.0, 'train_loss': 0.11429129918884862, 'epoch': 5.0})

In [10]:
MODEL_NAME  = "distilroberta-base"
NUM_LABELS  = 28
MAX_LENGTH  = 128
BATCH_SIZE  = 32
EPOCHS      = 5
LR          = 2e-5
OUTPUT_DIR  = "./go_emotions_model"
THRESHOLD   = 0.5

In [6]:
raw = load_dataset("go_emotions", "simplified")
print(raw)
print("\nSample:", raw["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

Sample: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


In [23]:
val_output = trainer_25.predict(tokenized_25["validation"])
val_probs = torch.sigmoid(torch.tensor(val_output.predictions)).numpy()
val_labels = val_output.label_ids.astype(int)

thresholds_25 = []
for i in range(NUM_LABELS_25):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.90, 0.05):
        preds = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds_25.append(best_t)
    print(f"{label_names_25[i]:20s}  threshold={best_t:.2f}  f1={best_f1:.3f}")

thresholds_25 = np.array(thresholds_25)

admiration            threshold=0.55  f1=0.735
amusement             threshold=0.50  f1=0.798
anger                 threshold=0.30  f1=0.511
annoyance             threshold=0.25  f1=0.408
approval              threshold=0.25  f1=0.406
caring                threshold=0.25  f1=0.467
confusion             threshold=0.45  f1=0.445
curiosity             threshold=0.25  f1=0.548
desire                threshold=0.15  f1=0.543
disappointment        threshold=0.20  f1=0.325
disapproval           threshold=0.15  f1=0.421
disgust               threshold=0.35  f1=0.449
embarrassment         threshold=0.35  f1=0.588
excitement            threshold=0.20  f1=0.335
fear                  threshold=0.15  f1=0.660
gratitude             threshold=0.80  f1=0.916
joy                   threshold=0.35  f1=0.580
love                  threshold=0.45  f1=0.791
nervousness           threshold=0.15  f1=0.438
optimism              threshold=0.20  f1=0.578
realization           threshold=0.25  f1=0.269
remorse      

In [24]:
from sklearn.metrics import classification_report

pred_output = trainer_25.predict(tokenized_25["test"])
probs = torch.sigmoid(torch.tensor(pred_output.predictions)).numpy()
preds = np.stack([
    (probs[:, i] > thresholds_25[i]).astype(int) for i in range(NUM_LABELS_25)
], axis=1)
labels = pred_output.label_ids.astype(int)

print(classification_report(labels, preds, target_names=label_names_25, zero_division=0))

                precision    recall  f1-score   support

    admiration       0.73      0.71      0.72       514
     amusement       0.78      0.86      0.82       264
         anger       0.44      0.53      0.48       198
     annoyance       0.35      0.42      0.38       320
      approval       0.39      0.46      0.43       351
        caring       0.40      0.47      0.43       135
     confusion       0.57      0.34      0.43       153
     curiosity       0.46      0.77      0.57       284
        desire       0.51      0.52      0.51        83
disappointment       0.32      0.36      0.34       151
   disapproval       0.32      0.61      0.42       267
       disgust       0.55      0.40      0.46       123
 embarrassment       0.80      0.22      0.34        37
    excitement       0.35      0.52      0.42       103
          fear       0.47      0.79      0.59        78
     gratitude       0.97      0.86      0.91       352
           joy       0.61      0.62      0.62  

In [25]:
import shutil, os

trainer_25.save_model(RETRAIN_25_DIR)
tokenizer.save_pretrained(RETRAIN_25_DIR)

# Save label names and thresholds alongside the model
np.save(f"{RETRAIN_25_DIR}/thresholds.npy", thresholds_25)
with open(f"{RETRAIN_25_DIR}/label_names.txt", "w") as f:
    f.write("\n".join(label_names_25))

DRIVE_SAVE_25 = "/content/drive/MyDrive/go_emotions_model_25"
if os.path.exists(DRIVE_SAVE_25):
    shutil.rmtree(DRIVE_SAVE_25)
shutil.copytree(RETRAIN_25_DIR, DRIVE_SAVE_25)
print(f"Saved to {DRIVE_SAVE_25}")
print(f"Files: {os.listdir(DRIVE_SAVE_25)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/go_emotions_model_25
Files: ['label_names.txt', 'checkpoint-6785', 'checkpoint-5428', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors', 'config.json', 'training_args.bin', 'thresholds.npy']


In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    encoding = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    multi_hot = np.zeros((len(batch["labels"]), NUM_LABELS), dtype=np.float32)
    for i, label_list in enumerate(batch["labels"]):
        for l in label_list:
            multi_hot[i][l] = 1.0
    encoding["labels"] = multi_hot.tolist()
    return encoding

tokenized = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)
tokenized.set_format("torch")
print("Labels shape check:", tokenized["train"][0]["labels"].shape)

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Labels shape check: torch.Size([28])


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # sets BCEWithLogitsLoss internally
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 82.1M


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > THRESHOLD).astype(int)
    labels = labels.astype(int)
    return {
        "f1_macro":    f1_score(labels, preds, average="macro",    zero_division=0),
        "f1_micro":    f1_score(labels, preds, average="micro",    zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    logging_steps=100,
    save_total_limit=2,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
from transformers import DataCollatorWithPadding

class MultiLabelDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features)
        # Ensure 'labels' are of type torch.float32
        if "labels" in batch:
            batch["labels"] = batch["labels"].to(torch.float32)
        return batch

# Instantiate the custom data collator
data_collator = MultiLabelDataCollator(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    data_collator=data_collator, # Add the custom data collator here
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,F1 Weighted
1,0.074510,0.087068,0.404479,0.569530,0.537617
2,0.067647,0.088198,0.444254,0.564121,0.546564
3,0.061628,0.089328,0.444104,0.564611,0.547724
4,0.059604,0.091844,0.446422,0.562239,0.544601
5,0.057852,0.092639,0.450639,0.563326,0.548484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6785, training_loss=0.06397962563162617, metrics={'train_runtime': 864.3114, 'train_samples_per_second': 251.125, 'train_steps_per_second': 7.85, 'total_flos': 7191345107404800.0, 'train_loss': 0.06397962563162617, 'epoch': 5.0})

In [22]:
results = trainer_25.evaluate(tokenized["test"])
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

ValueError: Target size (torch.Size([64, 28])) must be the same as input size (torch.Size([64, 25]))

In [13]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./go_emotions_model


In [15]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/go_emotions_model')
print("Saved to Google Drive")

Mounted at /content/drive


FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/go_emotions_model'

In [1]:
label_names = raw["train"].features["labels"].feature.names

def predict_emotions(text, threshold=THRESHOLD):
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    ).to("cuda")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    return [(label_names[i], round(float(probs[i]), 3)) for i in np.where(probs > threshold)[0]]

test_texts = [
    "I can't believe they did that, I'm absolutely furious!",
    "This made my whole week, thank you so much!",
    "I'm not sure how I feel about this...",
]

for text in test_texts:
    print(f"\nText: {text}")
    print(f"Emotions: {predict_emotions(text)}")

NameError: name 'raw' is not defined

In [1]:
# ── Reload model + tokenizer ──────────────────────────────────────────────────
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import load_dataset
from sklearn.metrics import f1_score
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/go_emotions_model"
NUM_LABELS = 28
MAX_LENGTH = 128
THRESHOLD  = 0.5

# Load
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.to("cuda")
model.eval()

# Reload tokenized test set
raw = load_dataset("go_emotions", "simplified")

def preprocess(batch):
    encoding = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    multi_hot = np.zeros((len(batch["labels"]), NUM_LABELS), dtype=np.float32)
    for i, label_list in enumerate(batch["labels"]):
        for l in label_list:
            multi_hot[i][l] = 1.0
    encoding["labels"] = multi_hot.tolist()
    return encoding

tokenized = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)
tokenized.set_format("torch")

label_names = raw["train"].features["labels"].feature.names

# Custom Data Collator to ensure float32 labels
class MultiLabelDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features)
        if "labels" in batch:
            batch["labels"] = batch["labels"].to(torch.float32)
        return batch

data_collator = MultiLabelDataCollator(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > THRESHOLD).astype(int)
    return {"f1_macro": f1_score(labels.astype(int), preds, average="macro", zero_division=0)}

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir=OUTPUT_DIR, report_to="none"),
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("Model loaded and ready with proper data_collator.")

Mounted at /content/drive


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Model loaded and ready with proper data_collator.


In [5]:
from sklearn.metrics import f1_score
import numpy as np

# Get validation logits
val_output = trainer.predict(tokenized["validation"])
val_probs = torch.sigmoid(torch.tensor(val_output.predictions)).numpy()
val_labels = val_output.label_ids.astype(int)

# Find optimal threshold per class
thresholds = []
for i in range(NUM_LABELS):
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.1, 0.9, 0.05):
        preds = (val_probs[:, i] > t).astype(int)
        f1 = f1_score(val_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    thresholds.append(best_t)
    print(f"{label_names[i]:20s}  threshold={best_t:.2f}  f1={best_f1:.3f}")

thresholds = np.array(thresholds)

admiration            threshold=0.30  f1=0.714
amusement             threshold=0.50  f1=0.787
anger                 threshold=0.50  f1=0.496
annoyance             threshold=0.15  f1=0.376
approval              threshold=0.30  f1=0.394
caring                threshold=0.20  f1=0.460
confusion             threshold=0.25  f1=0.485
curiosity             threshold=0.20  f1=0.537
desire                threshold=0.35  f1=0.534
disappointment        threshold=0.20  f1=0.308
disapproval           threshold=0.15  f1=0.420
disgust               threshold=0.35  f1=0.448
embarrassment         threshold=0.40  f1=0.607
excitement            threshold=0.10  f1=0.308
fear                  threshold=0.25  f1=0.656
gratitude             threshold=0.75  f1=0.918
grief                 threshold=0.50  f1=0.000
joy                   threshold=0.35  f1=0.560
love                  threshold=0.30  f1=0.788
nervousness           threshold=0.25  f1=0.312
optimism              threshold=0.30  f1=0.581
pride        

In [15]:
from sklearn.metrics import classification_report

pred_output = trainer.predict(tokenized["test"])
probs = torch.sigmoid(torch.tensor(pred_output.predictions)).numpy()
preds = (probs > THRESHOLD).astype(int)
labels = pred_output.label_ids.astype(int)

print(classification_report(labels, preds, target_names=label_names, zero_division=0))

                precision    recall  f1-score   support

    admiration       0.50      0.89      0.64       504
     amusement       0.68      0.95      0.79       264
         anger       0.30      0.73      0.43       198
     annoyance       0.22      0.68      0.33       320
      approval       0.24      0.70      0.36       351
        caring       0.25      0.62      0.36       135
     confusion       0.22      0.82      0.35       153
     curiosity       0.39      0.89      0.54       284
        desire       0.40      0.64      0.50        83
disappointment       0.16      0.52      0.25       151
   disapproval       0.26      0.73      0.38       267
       disgust       0.33      0.71      0.45       123
 embarrassment       0.45      0.46      0.45        37
    excitement       0.24      0.69      0.35       103
          fear       0.42      0.81      0.56        78
     gratitude       0.75      0.95      0.84       352
         grief       0.00      0.00      0.00  

In [10]:

label_counts = np.zeros(NUM_LABELS)
for example in tokenized["train"]:
    label_counts += example["labels"].numpy()

total = len(tokenized["train"])

# Raw weights
raw_weights = (total - label_counts) / (label_counts + 1e-6)

# Clip to max 10 — prevents rare classes from dominating
clipped_weights = np.clip(raw_weights, 1.0, 10.0)

class_weights = torch.tensor(clipped_weights, dtype=torch.float32).to("cuda")

print("Class weights (clipped):")
for i, name in enumerate(label_names):
    print(f"  {name:20s}  raw={raw_weights[i]:.1f}  clipped={clipped_weights[i]:.1f}")

Class weights (clipped):
  admiration            raw=9.5  clipped=9.5
  amusement             raw=17.6  clipped=10.0
  anger                 raw=26.7  clipped=10.0
  annoyance             raw=16.6  clipped=10.0
  approval              raw=13.8  clipped=10.0
  caring                raw=38.9  clipped=10.0
  confusion             raw=30.7  clipped=10.0
  curiosity             raw=18.8  clipped=10.0
  desire                raw=66.7  clipped=10.0
  disappointment        raw=33.2  clipped=10.0
  disapproval           raw=20.5  clipped=10.0
  disgust               raw=53.7  clipped=10.0
  embarrassment         raw=142.3  clipped=10.0
  excitement            raw=49.9  clipped=10.0
  fear                  raw=71.8  clipped=10.0
  gratitude             raw=15.3  clipped=10.0
  grief                 raw=562.8  clipped=10.0
  joy                   raw=28.9  clipped=10.0
  love                  raw=19.8  clipped=10.0
  nervousness           raw=263.7  clipped=10.0
  optimism              raw=26.5  

In [11]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights)(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilroberta-base",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)
model.to("cuda")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [13]:
RETRAIN_OUTPUT_DIR = "./go_emotions_model_weighted"

training_args = TrainingArguments(
    output_dir=RETRAIN_OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    logging_steps=100,
    save_total_limit=2,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:

from transformers import EarlyStoppingCallback
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    data_collator=data_collator # Added this to fix the Float vs Long error
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.380694,0.350811,0.394786
2,0.319673,0.316328,0.435967
3,0.281770,0.303997,0.470976
4,0.269247,0.301878,0.468936
5,0.246442,0.300963,0.472313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=6785, training_loss=0.3270286281757172, metrics={'train_runtime': 859.915, 'train_samples_per_second': 252.409, 'train_steps_per_second': 7.89, 'total_flos': 7191345107404800.0, 'train_loss': 0.3270286281757172, 'epoch': 5.0})